# Exam Countdown Planner (Agentic AI) Demo

This notebook demonstrates the execution of the agentic plan-act loop of the **Exam Countdown Planner**. It runs in-process using FastAPI's `TestClient` to make execution simple and self-contained.

The demo covers three scenarios designed to test double-depth vagueness resolution:
1. **Goal 1 (Broad Umbrella Category -> Conversational Rejection -> Auto-Expansion)**: The user provides a very broad category: `Math`. The agent sets the exam date but halts to ask the user for specific subjects, recommending Algebra and Geometry. Once the user replies, the agent automatically divides them into mastery subtopics and allocates them.
2. **Goal 2 (Category B Auto-Breakdown & Allocation)**: The user provides `Algebra and Rotational Motion`. Since these are Category B subjects, the agent automatically breaks them down into specific mastery subtopics and schedules them immediately.
3. **Goal 3 (Missed Day & Catch-Up Shuffle)**: Simulates missing a day in the broken-down schedule, showing how subtopics shift forward.

In [ ]:
import os
import json

# Override 'today' to make output deterministic and independent of execution date
os.environ['TODAY_OVERRIDE'] = '2026-08-24'

from fastapi.testclient import TestClient
from app.main import app

# Initialize the test client
client = TestClient(app)

def run_demo_step(session_id: str, message: str):
    print(f"\033[1;34m[USER MESSAGE]\033[0m: '{message}'")
    response = client.post("/chat", json={"session_id": session_id, "message": message})
    assert response.status_code == 200, f"API failed: {response.text}"
    data = response.json()
    
    print("\n\033[1;35m--- AGENT PLAN-ACT LOOP TRACE ---\033[0m")
    for step in data["trace"]:
        step_type = step["type"]
        if step_type == "assistant_thought":
            print(f"\033[1;30m[Thought]\033[0m: {step['content']}")
        elif step_type == "tool_call":
            print(f"\033[1;33m[Tool Call]\033[0m: {step['tool']}({step['args']})")
        elif step_type == "tool_result":
            print(f"\033[1;32m[Tool Result]\033[0m: {step['tool']} returned: {step['result']}")
        elif step_type == "final_answer":
            print(f"\033[1;36m[Final Answer]\033[0m: {step['content']}")
        elif step_type == "error":
            print(f"\033[1;31m[Error]\033[0m: {step['content']}")
            
    print("\n\033[1;35m--- SESSION MEMORY STATE ---\033[0m")
    state = data["state"]
    print(f"Exam Date: {state.get('exam_date')}")
    print(f"Days Remaining: {state.get('days_left')}")
    print(f"Completed Days: {state.get('completed_days')}")
    print(f"Missed Days: {state.get('missed_days')}")
    print("Day-by-Day Study Plan:")
    plan = state.get("day_plan", {})
    try:
        sorted_days = sorted(plan.items(), key=lambda x: int(x[0].split()[1]))
        for day, topics in sorted_days:
            print(f"  {day}: {topics}")
    except Exception:
        for day, topics in plan.items():
            print(f"  {day}: {topics}")
    print("\n" + "="*75 + "\n")

## Goal 1: Broad Umbrella Subjects & Subtopic Mastery Breakdown (Conversational Rejection & Auto-Expansion)

Here we test the double-depth vagueness resolution:
1. **Step A**: The user provides a very broad category: `Math`. The agent should set the exam date but halt and ask the user for specific subjects, recommending Algebra and Geometry.
2. **Step B**: The user replies `Algebra and Geometry`. Since these are Category B subjects, the agent should automatically divide them into concrete mastery subtopics (e.g. quadratic equations, straight line theorems) and call `allocate_topics` with those subtopics in one go!

In [ ]:
# Ensure a clean session state
client.post("/reset", json={"session_id": "demo_session_1"})

# 1. User starts with broad category 'Math'
run_demo_step(
    session_id="demo_session_1", 
    message="My exam is on 2026-09-15. I need to study Math."
)

# 2. User inputs the subjects in reply
run_demo_step(
    session_id="demo_session_1",
    message="Algebra and Geometry"
)

## Goal 2: Category B Auto-Breakdown & Allocation in One Go

Here the user directly inputs `Algebra and Rotational Motion`. Since these are Category B subjects, the agent should automatically break them down into specific mastery subtopics and execute the plan allocation immediately.

In [ ]:
# Ensure clean state for second session
client.post("/reset", json={"session_id": "demo_session_2"})

run_demo_step(
    session_id="demo_session_2", 
    message="My exam is on 2026-09-15. I need to study Algebra and Rotational Motion."
)

## Goal 3: Missed Day & Catch-Up Shuffle

Using the active plan from `demo_session_2` (where the subtopics are allocated), we report that we missed Day 3. The agent should mark Day 3 as missed and reshuffle the specific subtopics forward.

In [ ]:
run_demo_step(
    session_id="demo_session_2", 
    message="I missed Day 3, I was sick. Reshuffle my plan please."
)